#### Import and Load

In [158]:
import pandas as pd
import numpy as np

In [159]:
pd.set_option('display.max_columns', None)

df = pd.read_csv("../Data/Raw_Dataset/emi_prediction_dataset.csv")

C:\Users\hp\AppData\Local\Temp\ipykernel_14652\3391078597.py:3: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../Data/Raw_Dataset/emi_prediction_dataset.csv")


#### Fixing Wrong Data types

### Convert Age

In [160]:
df['age'] = pd.to_numeric(df['age'], errors='coerce')

### Convert monthly_salary

In [161]:
# Remove commas
df['monthly_salary'] = df['monthly_salary'].astype(str).str.replace(',', '')

# Convert to numeric
df['monthly_salary'] = pd.to_numeric(df['monthly_salary'], errors='coerce')

### Convert bank_balance

In [162]:
df['bank_balance'] = df['bank_balance'].astype(str).str.replace(',', '')
df['bank_balance'] = pd.to_numeric(df['bank_balance'], errors='coerce')

In [163]:
df.dtypes

age                       float64
gender                     object
marital_status             object
education                  object
monthly_salary            float64
employment_type            object
years_of_employment       float64
company_type               object
house_type                 object
monthly_rent              float64
family_size                 int64
dependents                  int64
school_fees               float64
college_fees              float64
travel_expenses           float64
groceries_utilities       float64
other_monthly_expenses    float64
existing_loans             object
current_emi_amount        float64
credit_score              float64
bank_balance              float64
emergency_fund            float64
emi_scenario               object
requested_amount          float64
requested_tenure            int64
emi_eligibility            object
max_monthly_emi           float64
dtype: object

#### Handle Missing Values

### Numerical Columns

In [164]:
num_cols = df.select_dtypes(include=['int64', 'float64']).columns

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

#### Categorical Columns

In [165]:
cat_cols = df.select_dtypes(include='object').columns

for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

## Cleaning

#### Gender

In [166]:
df['gender'] = df['gender'].astype(str).str.strip().str.lower()

In [167]:
df['gender'] = df['gender'].replace({
    'm': 'male',
    'male': 'male',
    'f': 'female',
    'female': 'female'
})

In [168]:
df['gender'].value_counts()

gender
male      242950
female    161850
Name: count, dtype: int64

In [169]:
df['gender'] = df['gender'].map({'male': 0, 'female': 1})

#### Age

In [170]:
df['age'] = pd.to_numeric(df['age'], errors='coerce')

df['age'] = df['age'].fillna(df['age'].median())

df['age'] = df['age'].astype(int)

#### Numeric Checking

In [171]:
df['monthly_salary'] = pd.to_numeric(df['monthly_salary'], errors='coerce')
df['bank_balance'] = pd.to_numeric(df['bank_balance'], errors='coerce')

## Verify Cleaning

In [172]:
print("Missing values after cleaning:\n")
print(df.isnull().sum().sum())

Missing values after cleaning:

0


In [173]:
for col in df.select_dtypes(include='object').columns:
    print(f"\nColumn: {col}")
    print("-" * 40)
    print(df[col].value_counts().head(10))


Column: marital_status
----------------------------------------
marital_status
Married    307837
Single      96963
Name: count, dtype: int64

Column: education
----------------------------------------
education
Graduate         183419
Post Graduate    100314
High School       60732
Professional      60335
Name: count, dtype: int64

Column: employment_type
----------------------------------------
employment_type
Private          283099
Government        81167
Self-employed     40534
Name: count, dtype: int64

Column: company_type
----------------------------------------
company_type
Large Indian    121139
MNC             101409
Mid-size        101301
Startup          60706
Small            20245
Name: count, dtype: int64

Column: house_type
----------------------------------------
house_type
Rented    161601
Own       142307
Family    100892
Name: count, dtype: int64

Column: existing_loans
----------------------------------------
existing_loans
No     243227
Yes    161573
Name: count,

In [174]:
for col in df.select_dtypes(include='object').columns:
    print(f"{col}: {df[col].nunique()} unique values")

marital_status: 2 unique values
education: 4 unique values
employment_type: 3 unique values
company_type: 5 unique values
house_type: 3 unique values
existing_loans: 2 unique values
emi_scenario: 5 unique values
emi_eligibility: 3 unique values


In [175]:
for col in df.select_dtypes(include='object').columns:
    if df[col].nunique() > 5:
        print(f"{col} → {df[col].nunique()} unique values ⚠️")

In [176]:
for col in df.select_dtypes(include='object').columns:
    raw_unique = df[col].nunique()
    cleaned_unique = df[col].str.lower().str.strip().nunique()
    
    if raw_unique != cleaned_unique:
        print(f"{col}: {raw_unique} → {cleaned_unique} after cleaning")

## Encoding Categorical Variables

#### Label Encoding

In [177]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for col in cat_cols:
    df[col] = le.fit_transform(df[col])

## Split Features and Targets


#### Classification Target

In [178]:
X_class = df.drop(['emi_eligibility', 'max_monthly_emi'], axis=1)
y_class = df['emi_eligibility']

#### Regression Target

In [179]:
X_reg = df.drop(['max_monthly_emi', 'emi_eligibility'], axis=1)
y_reg = df['max_monthly_emi']

#### Identify Categorical Columns

In [180]:
cat_cols = X.select_dtypes(include='object').columns
num_cols = X.select_dtypes(exclude='object').columns

#### Apply One-Hot Encoding

In [181]:
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

#### Encode Target

In [182]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_class = le.fit_transform(y_class)

## Handling Imbalance

In [183]:
# from imblearn.over_sampling import SMOTE

# smote = SMOTE(random_state=42)

# X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train_class)

In [184]:
corr = df.corr(numeric_only=True)
corr['max_monthly_emi'].sort_values(ascending=False)

max_monthly_emi           1.000000
groceries_utilities       0.484695
bank_balance              0.456028
travel_expenses           0.440948
emergency_fund            0.414036
other_monthly_expenses    0.382055
monthly_salary            0.378284
credit_score              0.228814
education                 0.195277
marital_status            0.055107
years_of_employment       0.028887
requested_tenure          0.001006
company_type             -0.000589
age                      -0.000848
emi_scenario             -0.001163
gender                   -0.001844
requested_amount         -0.002034
dependents               -0.062266
family_size              -0.062266
employment_type          -0.065718
monthly_rent             -0.169423
school_fees              -0.215234
current_emi_amount       -0.242434
college_fees             -0.258842
house_type               -0.277618
existing_loans           -0.384001
emi_eligibility          -0.593031
Name: max_monthly_emi, dtype: float64

## Encoding

#### Binary-> Map

In [185]:
df['marital_status'] = df['marital_status'].map({
    'Single': 0,
    'Married': 1
})

df['existing_loans'] = df['existing_loans'].map({
    'No': 0,
    'Yes': 1
})

#### Multi Category -> One Hot

In [186]:
multi_cols = [
    'education',
    'employment_type',
    'company_type',
    'house_type',
    'emi_scenario'
]

df = pd.get_dummies(df, columns=multi_cols, drop_first=True)

#### Target Encoding

In [187]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['emi_eligibility'] = le.fit_transform(df['emi_eligibility'])

## 3. Feature Split

In [188]:
X = df.drop(['emi_eligibility', 'max_monthly_emi'], axis=1)

y_class = df['emi_eligibility']
y_reg = df['max_monthly_emi']

## 4. Train-Test Split

In [189]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train_class, y_test_class, y_train_reg, y_test_reg = train_test_split(
    X, y_class, y_reg,
    test_size=0.2,
    random_state=42,
    stratify=y_class
)

In [190]:
median_values = X.median()
import joblib
joblib.dump(median_values, "../models/median_values.pkl")

['../models/median_values.pkl']

In [191]:
df.to_csv("../Data/processed_dataset/emi_final.csv", index=False)